<a href="https://colab.research.google.com/github/edik06031-rgb/DTA_2026/blob/main/ML/ml_practice_LR%26Cl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практика: лінійна регресія та класифікація

Це тренувальний блокнот для закріплення базового циклу ML. Завдання **нескладні** й повторюють кроки з основного тьюторіалу — тільки тепер усе робиш **сам**.

**Дві задачі на двох нових наборах даних:**
- **Задача A (регресія):** передбачити **зарплату** працівника.
- **Задача B (класифікація):** передбачити, чи **складе студент іспит** (так/ні).

**Як працювати:**
1. Запусти комірку «Підготовка даних» нижче — вона все налаштує.
2. Іди по кроках. Там, де стоїть `# TODO`, — впиши свій код.
3. Підказки є під кожним кроком.

> 💡 Усі потрібні інструменти ти вже бачив: `train_test_split`, `LinearRegression`, `DecisionTreeClassifier`, `.fit()`, `.predict()`, метрики. Тримай той блокнот поруч як шпаргалку.

---

## 🔧 Підготовка даних (просто запусти)

In [16]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 20)

# ---------- Дані A: ЗАРПЛАТИ (для регресії) ----------
N = 800
experience = np.random.randint(0, 31, N)                      # стаж, років
age        = (22 + experience + np.random.randint(0, 12, N)).clip(22, 64)  # вік
education  = np.random.choice([12, 15, 16, 18, 20], N,        # років освіти
                              p=[.2, .15, .35, .2, .1])
english    = np.random.randint(1, 6, N)                       # рівень англ. 1..5

salary = (8000                       # базова ставка, грн
          + experience * 900         # за кожен рік стажу
          + education  * 600         # за рік освіти
          + english    * 1500        # за рівень англійської
          + np.random.normal(0, 3000, N)   # шум: усе інше
         ).clip(8000, None)

salary_df = pd.DataFrame({
    "experience": experience, "age": age,
    "education": education, "english": english,
    "salary": salary.round(0).astype(int),
})

# ---------- Дані B: СТУДЕНТИ (для класифікації) ----------
M = 800
study     = np.random.normal(12, 5, M).clip(0, 30)            # годин навчання/тиждень
attendance= np.random.normal(78, 15, M).clip(30, 100)        # відвідуваність, %
prev_score= np.random.normal(65, 18, M).clip(0, 100)         # бал за минулий іспит
sleep     = np.random.normal(7, 1.2, M).clip(4, 10)          # годин сну

score_logit = (0.12*study + 0.04*attendance + 0.05*prev_score
               + 0.3*sleep - 9 + np.random.normal(0, 1.2, M))
passed = (score_logit > 0).astype(int)                        # 1 = склав, 0 = ні

students_df = pd.DataFrame({
    "study": study.round(1), "attendance": attendance.round(0).astype(int),
    "prev_score": prev_score.round(0).astype(int), "sleep": sleep.round(1),
    "passed": passed,
})

print("✅ Дані готові.")
print("Зарплати:", salary_df.shape, "| Студенти:", students_df.shape)
print("Частка тих, хто склав іспит:", f"{students_df['passed'].mean():.0%}")

✅ Дані готові.
Зарплати: (800, 5) | Студенти: (800, 5)
Частка тих, хто склав іспит: 69%


---
# 🟦 Задача A. Регресія: передбачаємо зарплату

Дані у таблиці `salary_df`. Ознаки: `experience` (стаж), `age` (вік), `education` (років освіти), `english` (рівень англійської 1–5). Ціль: `salary` (зарплата, грн).

Мета — навчити модель передбачати зарплату і **пояснити**, що на неї впливає.

### Крок A1. Подивись на дані
Виведи перші рядки таблиці й описову статистику. Це звичка №1 перед будь-яким навчанням.

*Підказка:* `salary_df.head()` і `salary_df.describe()`.

In [17]:
# TODO: виведи перші рядки salary_df
salary_df.head()

# TODO: виведи describe()
salary_df.describe().round(2)

,experience,age,education,english,salary
count,800.00,800.00,800.00,800.00,800.00
mean,15.42,42.75,15.86,3.03,36028.93
std,9.33,9.92,2.41,1.43,9242.04
min,0.00,22.00,12.00,1.00,12917.00
25%,7.00,35.00,15.00,2.00,28550.75
50%,16.00,43.00,16.00,3.00,36102.50
75%,24.00,51.00,18.00,4.00,43545.50
max,30.00,62.00,20.00,5.00,55947.00


### Крок A2. Признач ознаки (X) і ціль (y), поділи на train / test
- `X` — усі стовпці, КРІМ `salary`.
- `y` — стовпець `salary`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`.

*Підказка:* `X = salary_df[["experience", "age", "education", "english"]]`,
`y = salary_df["salary"]`, далі `train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)`.

In [4]:
from sklearn.model_selection import train_test_split

# TODO: створи X та y
X = salary_df[["experience", "age", "education", "english"]]
y = salary_df["salary"]

# TODO: поділи на X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)


print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 640 | Test: 160


### Крок A3. Навчи лінійну регресію
Згадай цикл: **створити → `.fit(X_train, y_train)`**.

*Підказка:* `from sklearn.linear_model import LinearRegression`, далі `model = LinearRegression()` і `model.fit(...)`.

In [5]:
from sklearn.linear_model import LinearRegression

# TODO: створи та навчи модель
model = LinearRegression()
model.fit(X_train, y_train)


LinearRegression()

### Крок A4. Зроби передбачення й оціни якість
- Передбач на `X_test`.
- Порахуй **MAE** та **R²**.

*Підказка:* `y_pred = model.predict(X_test)`; `mean_absolute_error(y_test, y_pred)`;
`r2_score(y_test, y_pred)`.

In [6]:
from sklearn.metrics import mean_absolute_error, r2_score

# TODO: передбач y_pred
y_pred = model.predict(X_test)


# TODO: порахуй і виведи MAE та R²
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE = {mae:.1f} грн (у середньому помиляемося на скільки) ")
print(f"R² = {r2:.3f} модель пояснює {r2*100:.1f}% розкиду ціни")

MAE = 2545.3 грн (у середньому помиляемося на скільки) 
R² = 0.881 модель пояснює 88.1% розкиду ціни


### Крок A5. 🔑 Інтерпретуй коефіцієнти
Дістань коефіцієнти моделі й скажи словами, яка ознака найсильніше підвищує зарплату.

*Підказка:* `model.coef_` і `model.intercept_`. Зістав назви з `X.columns`.

In [7]:
# TODO: побудуй таблицю "ознака — коефіцієнт" і відсортуй
coefs = pd.DataFrame({
    "features" : X.columns,
    "coef": model.coef_.round(2)

}).sort_values("coef", ascending=False)
print(f"Базова зарплата : {model.intercept_:.2f} грн\n")
print("Як кожна ознака впливає на зарплату")
coefs

Базова зарплата : 6850.03 грн

Як кожна ознака впливає на зарплату


,features,coef
3,english,1597.45
0,experience,872.48
2,education,609.59
1,age,31.15


✍️ **Запиши відповідь словами** (просто текстом у цій комірці, подвійний клік):
> Найсильніше на зарплату впливає ознака ___english_, бо англійська признана моваі не тільки в спілкуванні у світі, але і в більшості компаній по усьомц світу і гарні знання англійскої це додатковий плюс(бонус) у роботі.
Окрім цього за інших однакових умов підвищення рівня англійської на 1 пункт пов'язане зі збільшенням очікуваної зарплати приблизно на 1597 грн.

### Крок A6. Передбач зарплату для нового працівника
Створи одного працівника й передбач його зарплату: стаж 5, вік 30, освіта 16, англійська 4.

*Підказка:* зроби `pd.DataFrame([{...}])` з тими самими назвами стовпців і передай у `model.predict(...)`.

In [8]:
# TODO: створи new_employee і передбач зарплату

# Новий працівник
new_employee = pd.DataFrame([{
    'experience': 5,
    'age': 30,
    'education': 16,
    'english': 4
}])

# Прогноз зарплати
pred_salary = model.predict(new_employee)

print(f'Передбачена зарплата: {pred_salary[0]:.2f}')

Передбачена зарплата: 28290.19


---
# 🟩 Задача B. Класифікація: чи складе студент іспит

Дані у таблиці `students_df`. Ознаки: `study` (годин навчання/тиждень), `attendance` (відвідуваність %), `prev_score` (бал за минулий іспит), `sleep` (годин сну). Ціль: `passed` (1 = склав, 0 = ні).

### Крок B1. Подивись на дані
Виведи перші рядки й перевір баланс класів: яка частка студентів склала іспит?

*Підказка:* `students_df.head()` і `students_df["passed"].mean()`.

In [9]:
# TODO: head() і частка тих, хто склав
students_df.head()
pass_rate = students_df["passed"].mean()

print(f'Частка студентів, які склали іспит: {pass_rate:.2%}')

print(students_df.head())



Частка студентів, які склали іспит: 69.38%
   study  attendance  prev_score  sleep  passed
0   11.2          76          94    5.7       1
1   18.5          72          49    9.4       1
2   15.7          71          59    7.9       0
3   16.1         100          88    5.1       1
4    9.9          92          72    7.0       1


### Крок B2. X, y і поділ на train / test
- `X` — усе, крім `passed`. `y` — `passed`.
- Додай `stratify=y`, щоб пропорція класів збереглася.

*Підказка:* `train_test_split(Xs, ys, test_size=0.2, random_state=RANDOM_STATE, stratify=ys)`.

In [10]:
# TODO: Xs, ys та поділ на Xs_train, Xs_test, ys_train, ys_test
# Ознаки та цільова змінна
Xs = students_df.drop(columns=['passed'])
ys = students_df['passed']

# Поділ на train / test зі збереженням пропорції класів
X_train, X_test, y_train, y_test = train_test_split(
    Xs,
    ys,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=ys
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (640, 4)
Test shape: (160, 4)


### Крок B3. Навчи дерево рішень
Використай `DecisionTreeClassifier` з `max_depth=3` (щоб було просте й читабельне) і `random_state=RANDOM_STATE`.

*Підказка:* `from sklearn.tree import DecisionTreeClassifier`.

In [11]:
from sklearn.tree import DecisionTreeClassifier

# TODO: створи та навчи дерево

# Створення моделі
tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE
)

# Навчання
tree_model.fit(X_train, y_train)

print("Модель успішно навчена!")

Модель успішно навчена!


### Крок B4. Передбач і оціни
- Передбач на `Xs_test`.
- Порахуй **accuracy** і побудуй **матрицю плутанини**.

*Підказка:* `accuracy_score(ys_test, ys_pred)` та `confusion_matrix(ys_test, ys_pred)`.

In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix

# TODO: передбач ys_pred, порахуй accuracy та матрицю плутанини

# Прогноз на тестовій вибірці
ys_pred = tree_model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, ys_pred)
print(f'Accuracy: {acc:.4f}')

# Матриця плутанини
cm = confusion_matrix(y_test, ys_pred)
print('\nConfusion Matrix:')
print(cm)

Accuracy: 0.7562

Confusion Matrix:
[[25 24]
 [15 96]]


### Крок B5. Що найбільше впливає на результат?
Виведи важливість ознак дерева й назви найважливішу.

*Підказка:* `tree.feature_importances_`, зістав із `Xs.columns`.

In [13]:
# TODO: таблиця "ознака — важливість", відсортована за спаданням
# Важливість ознак
feature_importance = pd.Series(
    tree_model.feature_importances_,
    index=Xs.columns
).sort_values(ascending=False)

print("Важливість ознак:")
print(feature_importance)

# Найважливіша ознака
top_feature = feature_importance.idxmax()


print(f"\nНайважливіша ознака: {top_feature}")
print(f"Важливість: {feature_importance.max():.4f}")

Важливість ознак:
prev_score    0.377026
study         0.325006
attendance    0.195933
sleep         0.102035
dtype: float64

Найважливіша ознака: prev_score
Важливість: 0.3770


✍️ **Відповідь словами:**
> Найбільше на складання іспиту впливає ___.

### Крок B6. Передбач для нового студента
Студент: навчання 15 год, відвідуваність 85%, минулий бал 70, сон 7.5.
Виведи і рішення (`predict`), і **ймовірність** скласти (`predict_proba`).

*Підказка:* `predict_proba(...)[0, 1]` — це ймовірність класу «склав».

In [19]:
# TODO: створи new_student, виведи рішення та ймовірність
# Новий студент
new_student = pd.DataFrame([{
    'study': 15,
    'attendance': 85,
    'prev_score': 70,
    'sleep': 7.5
}])

# Передбачення класу
prediction = tree_model.predict(new_student)[0]

# Ймовірність скласти іспит
probability = tree_model.predict_proba(new_student)[0, 1]

print(f'Рішення моделі (0=не склав, 1=склав): {prediction}')
print(f'Ймовірність скласти іспит: {probability:.2%}')

Рішення моделі (0=не склав, 1=склав): 1
Ймовірність скласти іспит: 88.05%


---
# ⭐ Бонус (необов'язково, але корисно)

1. **Перевір на перенавчання.** Для дерева зі Задачі B порахуй accuracy окремо на `Xs_train` і на `Xs_test`. Великий розрив = зубріння. Потім спробуй `max_depth=10` — розрив зросте?
2. **Сильніша модель.** Навчи `RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)` на тих самих даних і порівняй accuracy з деревом.
3. **Прибери ознаку.** У Задачі A прибери `experience` з `X`, перенавчи й подивись, як впаде R². Який висновок про важливість стажу?

In [15]:
# Місце для бонусних експериментів.  Нема сил для бонусних єкспериментів)

---
# 🧠 Питання на розуміння (без коду)

Дай собі відповідь словами (правильні — у блокноті з розв'язками):
1. Чому ми оцінюємо модель на `X_test`, а не на `X_train`?
2. Задача «передбачити кількість проданих квитків» — це регресія чи класифікація? А «спам / не спам»?
3. Що означає R² = 0.85 простими словами?
4. Чому accuracy може бути оманливою, якщо лише 5% студентів провалюють іспит?
5. Коефіцієнт `english = +1500`. Як прочитати це вголос для керівника?

> 🎯 Якщо впорався із задачами A і B без підглядання — ти впевнено володієш базовим циклом ML. Вітаю!

1. Чому ми оцінюємо модель на X_test, а не на X_train?

Тому що на X_train модель уже навчалася і могла запам'ятати дані. X_test показує, наскільки добре модель працює на нових даних, яких вона раніше не бачила.

2. «Передбачити кількість проданих квитків» — це регресія чи класифікація? А «спам / не спам»?
Передбачити кількість проданих квитків — регресія, тому що прогнозується число.
«Спам / не спам» — класифікація, тому що потрібно вибрати один із класів.
3. Що означає R² = 0.85 простими словами?

Це означає, що модель пояснює приблизно 85% змін у даних. Тобто її прогнози досить добрі і відповідають реальним значенням.

4. Чому accuracy може бути оманливою, якщо лише 5% студентів провалюють іспит?

Якщо модель завжди прогнозує «склав», вона матиме близько 95% accuracy, але не знайде жодного студента, який провалив іспит. Тому однієї accuracy недостатньо для оцінки таких даних.

5. Коефіцієнт english = +1500. Як прочитати це вголос для керівника?

За інших однакових умов підвищення рівня англійської на 1 пункт пов'язане зі збільшенням очікуваної зарплати приблизно на 1500 грн.